In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import scienceplots
from jax import vmap
from src.fdm import (
    ECirreMechanismFDMSolver,
    EMechanismFDMSolver,
    SecondOrderECirreFDMSolverBackwardImplicit,
    SecondOrderECirreFDMSolverExplicit,
)
from src.params import (
    ECirreMechanismFDMParams,
    EMechanismFDMParams,
    SecondOrderECirreMechanismFDMParams,
)
from src.plotting import plot_e_histograms, plot_ec_irre_histograms
from src.voltammetry import CyclicDC

plt.style.use("science")

# Electron Only Reaction

## Comparison with analytical results

In [ ]:
voltammetry = CyclicDC()

fdm_solver = EMechanismFDMSolver(voltammetry)

params = EMechanismFDMParams(
    alpha=jnp.array(0.7),
    K0=jnp.array(1.0),
    E0=jnp.array(0.0),
    dB=jnp.array(1.0),
)

current = fdm_solver.solve(params)

plt.figure(figsize=(10, 6))
plt.plot(fdm_solver.applied_potentials, current)
plt.axhline(
    y=-0.496 * jnp.sqrt(params.alpha) * jnp.sqrt(voltammetry.sigma),
    linestyle="--",
    c="red",
)
plt.axvline(
    x=(jnp.log(params.K0 / jnp.sqrt(params.alpha * voltammetry.sigma)) - 0.78)
    / params.alpha,
    linestyle="--",
    c="red",
)
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.ylabel("Current")
plt.xlabel("Applied Potential")
plt.show()


## Effects of each parameter


In [ ]:
voltammetry = CyclicDC()

fdm_solver = EMechanismFDMSolver(voltammetry)

base_params = EMechanismFDMParams(
    alpha=jnp.array(0.6), K0=jnp.array(1.0), E0=jnp.array(2.0), dB=jnp.array(0.5)
)

fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(16, 10), sharex=True, sharey=True)

# Alpha Varying

alpha_range = jnp.linspace(0.3, 0.7, 5)
alpha_params = EMechanismFDMParams(
    alpha=alpha_range,
    K0=jnp.full_like(alpha_range, base_params.K0),
    E0=jnp.full_like(alpha_range, base_params.E0),
    dB=jnp.full_like(alpha_range, base_params.dB),
)

currents = vmap(fdm_solver.solve)(alpha_params)

for val, current in zip(alpha_range, currents):
    ax[0, 0].plot(fdm_solver.applied_potentials, current, label=val)
ax[0, 0].xaxis.set_inverted(True)
ax[0, 0].yaxis.set_inverted(True)
ax[0, 0].set_title(r"$\alpha$")
ax[0, 0].legend()

# K0 Varying
K0_range = jnp.array([1.0, 5.0, 10.0, 20.0, 40.0, 50.0])
K0_params = EMechanismFDMParams(
    alpha=jnp.full_like(K0_range, base_params.alpha),
    E0=jnp.full_like(K0_range, base_params.E0),
    dB=jnp.full_like(K0_range, base_params.dB),
    K0=K0_range,
)

currents = vmap(fdm_solver.solve)(K0_params)

for val, current in zip(K0_range, currents):
    ax[0, 1].plot(fdm_solver.applied_potentials, current, label=f"{val:.0f}")
ax[0, 1].xaxis.set_inverted(True)
ax[0, 1].yaxis.set_inverted(True)
ax[0, 1].set_title(r"$K_0$")
ax[0, 1].legend()

# E0 Varying
E0_range = jnp.linspace(-2.0, 2.0, 5)
E0_params = EMechanismFDMParams(
    alpha=jnp.full_like(E0_range, base_params.alpha),
    E0=E0_range,
    dB=jnp.full_like(E0_range, base_params.dB),
    K0=jnp.full_like(E0_range, base_params.K0),
)

currents = vmap(fdm_solver.solve)(E0_params)

for val, current in zip(E0_range, currents):
    ax[1, 0].plot(fdm_solver.applied_potentials, current, label=val)

ax[1, 0].xaxis.set_inverted(True)
ax[1, 0].yaxis.set_inverted(True)
ax[1, 0].set_title(r"$E_0$")
ax[1, 0].legend()

# dB Varying
dB_range = jnp.array([0.1, 0.5, 1.0, 2.0, 5.0])

dB_params = EMechanismFDMParams(
    alpha=jnp.full_like(dB_range, base_params.alpha),
    E0=jnp.full_like(dB_range, base_params.E0),
    dB=dB_range,
    K0=jnp.full_like(dB_range, base_params.K0),
)

currents = vmap(fdm_solver.solve)(dB_params)

for val, current in zip(dB_range, currents):
    ax[1, 1].plot(fdm_solver.applied_potentials, current, label=val)

ax[1, 1].xaxis.set_inverted(True)
ax[1, 1].yaxis.set_inverted(True)
ax[1, 1].set_title(r"$d_B$")
ax[1, 1].legend()

plt.show()


## Sampling Histograms

In [ ]:
mchmc = np.load("./data/E_MCHMC_CyclicDC.npz")
rw = np.load("./data/E_MetropolisHastings_CyclicDC.npz")

datasets = {
    "MCHMC": {
        "alpha": mchmc["alpha"].flatten(),
        "K0": mchmc["K0"].flatten(),
        "E0": mchmc["E0"].flatten(),
        "dB": mchmc["dB"].flatten(),
    },
    "RW": {
        "alpha": rw["alpha"].flatten(),
        "K0": rw["K0"].flatten(),
        "E0": rw["E0"].flatten(),
        "dB": rw["dB"].flatten(),
    },
}

plot_e_histograms(
    datasets,
    heading="Posterior Distribution for Electrode-only Reaction using Cyclic Voltammetry",
)

In [ ]:
mchmc = np.load("./data/E_MCHMC_CyclicDC.npz")
rw = np.load("./data/E_MetropolisHastings_CyclicDC.npz")

datasets = {
    "MCHMC": {
        "alpha": mchmc["alpha"].flatten(),
        "K0": mchmc["K0"].flatten(),
        "E0": mchmc["E0"].flatten(),
        "dB": mchmc["dB"].flatten(),
    },
    "RW": {
        "alpha": rw["alpha"].flatten(),
        "K0": rw["K0"].flatten(),
        "E0": rw["E0"].flatten(),
        "dB": rw["dB"].flatten(),
    },
}

plot_e_histograms(
    datasets,
    heading="Posterior Distribution for Electrode-only Reaction using Cyclic Voltammetry",
)


# First-Order Chemical Kinetic Mechanism

In [ ]:
mchmc = np.load("./data/ECirre_MCHMC_CyclicDC.npz")
rw = np.load("./data/ECirre_MetropolisHastings_CyclicDC.npz")
datasets = {
    "MCHMC": {
        "alpha": mchmc["alpha"].flatten(),
        "K0": mchmc["K0"].flatten(),
        "Kplus": mchmc["Kplus"].flatten(),
        "Kminus": mchmc["Kminus"].flatten(),
        "E0": mchmc["E0"].flatten(),
        "dB": mchmc["dB"].flatten(),
    },
    "RW": {
        "alpha": rw["alpha"].flatten(),
        "K0": rw["K0"].flatten(),
        "Kplus": rw["Kplus"].flatten(),
        "Kminus": rw["Kminus"].flatten(),
        "E0": rw["E0"].flatten(),
        "dB": rw["dB"].flatten(),
    },
}

plot_ec_irre_histograms(
    datasets,
    # heading="Posterior Distribution for Electrode Irreducible Reaction using Cyclic Voltammetry",
)


# Second-Order Chemical Kinetic Mechanism

In [ ]:
voltammetry = CyclicDC()
# fdm_solver = SecondOrderECirreFDMSolverExplicit(voltammetry, h=1e-3, dtheta=5e-2)
fdm_solver = SecondOrderECirreFDMSolverBackwardImplicit(voltammetry)


base_params = SecondOrderECirreMechanismFDMParams(
    alpha=jnp.array(0.6),
    K0=jnp.array(10.0),
    Kplus=jnp.array(5.0),
    Kminus=jnp.array(1.0),
    dB=jnp.array(1.2),
    dY=jnp.array(0.8),
    dZ=jnp.array(0.6),
    E0=jnp.array(1.0),
)

print(base_params.dB, base_params.dY, base_params.dZ)

base_current = fdm_solver.solve(base_params)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5), sharey=True, sharex=True)

dB_range = jnp.array([0.2, 0.5, 0.8, 1.0, 1.2, 1.5])

dB_params = SecondOrderECirreMechanismFDMParams(
    alpha=jnp.full_like(dB_range, base_params.alpha),
    K0=jnp.full_like(dB_range, base_params.K0),
    Kplus=jnp.full_like(dB_range, base_params.Kplus),
    Kminus=jnp.full_like(dB_range, base_params.Kminus),
    dB=dB_range,
    dY=jnp.full_like(dB_range, base_params.dY),
    dZ=jnp.full_like(dB_range, base_params.dZ),
    E0=jnp.full_like(dB_range, base_params.E0),
)

dB_current = vmap(fdm_solver.solve)(dB_params)

for dB_c, dB in zip(dB_current, dB_range):
    ax1.plot(fdm_solver.applied_potentials, dB_c, label=dB)


dY_range = jnp.array([0.2, 0.5, 0.8, 1.0, 1.2, 1.5])

dY_params = SecondOrderECirreMechanismFDMParams(
    alpha=jnp.full_like(dY_range, base_params.alpha),
    K0=jnp.full_like(dY_range, base_params.K0),
    Kplus=jnp.full_like(dY_range, base_params.Kplus),
    Kminus=jnp.full_like(dY_range, base_params.Kminus),
    dB=jnp.full_like(dY_range, base_params.dB),
    dY=dY_range,
    dZ=jnp.full_like(dY_range, base_params.dZ),
    E0=jnp.full_like(dY_range, base_params.E0),
)

dY_current = vmap(fdm_solver.solve)(dY_params)

for dY_c, dY in zip(dY_current, dY_range):
    ax2.plot(fdm_solver.applied_potentials, dY_c, label=dY)

ax2.legend()

dZ_range = jnp.array([0.2, 0.5, 0.8, 1.0, 1.2, 1.5])

dZ_params = SecondOrderECirreMechanismFDMParams(
    alpha=jnp.full_like(dZ_range, base_params.alpha),
    K0=jnp.full_like(dZ_range, base_params.K0),
    Kplus=jnp.full_like(dZ_range, base_params.Kplus),
    Kminus=jnp.full_like(dZ_range, base_params.Kminus),
    dB=jnp.full_like(dZ_range, base_params.dB),
    dY=jnp.full_like(dZ_range, base_params.dY),
    dZ=dZ_range,
    E0=jnp.full_like(dZ_range, base_params.E0),
)

dZ_current = vmap(fdm_solver.solve)(dZ_params)

for dZ_c, dZ in zip(dZ_current, dZ_range):
    ax3.plot(fdm_solver.applied_potentials, dZ_c, label=dZ)

ax3.legend()

plt.show()


In [ ]:
plt.plot(jnp.abs(base_current[-50:] - dY_current[0][-50:]))
plt.plot(jnp.abs(base_current[-50:] - dY_current[1][-50:]))
plt.plot(jnp.abs(base_current[-50:] - dY_current[2][-50:]))
plt.plot(jnp.abs(base_current[-50:] - dY_current[3][-50:]))
plt.plot(jnp.abs(base_current[-50:] - dY_current[4][-50:]))
plt.yscale("log")
plt.show()